# Matelda Pipeline

As data-driven applications gain popularity, ensuring high data quality is a growing concern. This requirement involves not only the quality of primary data sources but also external data sources used for data enrichment purposes. Yet, data cleaning techniques are limited to treating one table at a time. A table-by-table application of such methods is cumbersome, because these methods either require previous knowledge about constraints or often require labor-intensive configurations and manual labeling for each individual table. As a result, they hardly scale beyond a few tables and miss the chance for optimizing the cleaning process. To tackle these issues, we introduce a novel semi-supervised error detection approach, Matelda, that organizes a given set of tables by folding their cells with regard to domain and quality similarity to facilitate user supervision. The idea is to identify groups of data cells across all tables that can benefit from the same user label. For this purpose, we identify a feature embedding that makes cell values comparable across many different tables. Experimental evaluations demonstrate that Matelda outperforms various configurations of existing single-table cleaning methodologies in cleaning multiple tables at a time, in particular when the ratio of labeling budget to number of tables is very low.

For more information about Matelda, we recommend you to read the [Paper](https://openproceedings.org/2025/conf/edbt/paper-98.pdf) and view the corresponding Code on [GitHub](https://github.com/LUH-DBS/Matelda).

# Initialization of Matelda

The initialization function in Matelda sets up the necessary configurations, directories, and variables, including paths for input data, output, logs, and results. It prepares the environment for the error detection process, ensuring all required directories exist and the configuration parameters are loaded correctly.

In [1]:
from pipeline_functions import init, domain_based_folding, quality_based_folding, loading_columns_grouping_results
import multiprocessing
import os 
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

In [2]:
!conda install ipywidgets -y

Channels:
 - defaults
 - conda-forge
Platform: linux-64
Solving environment: 

done

# All requested packages already installed.



In [3]:
# Initialization of Matelda

# Adjust the example configuration as needed
configs = {
    "EXPERIMENTS": {
        "labeling_budget": 27700,
        "exp_name": "test_edbt",
        "n_cores": 128,
        "save_mediate_res_on_disk": 1,
        "final_result_df": False,
    },
    "DIRECTORIES": {
        "sandbox_dir": "datasets",
        "tables_dir": "Quintet",
        "output_dir": "output_quintet/output_quintet",
        "results_dir": "results",
        "logs_dir": "logs",
        "aggregated_lake_path": "aggregated_lake",
        "dirty_files_name": "dirty.csv",
        "clean_files_name": "clean.csv",
    },
    "TABLE_GROUPING": {
        "tg_enabled": 1,
        "tg_res_available": 0,
        "tg_method": "bert",
    },
    "COLUMN_GROUPING": {
        "cg_enabled": 1,
        "cg_res_available": 0,
        "min_num_labes_per_col_cluster": 2,
        "cg_clustering_alg": "hac",
    },
    "CELL_GROUPING": {
        "cell_feature_generator_enabled": 1,
        "cell_clustering_alg": "km",
        "cell_clustering_res_available": 0,
        "classification_mode": 1,
        "labels_per_cell_group": 1,
    },
    "RAHA": {
        "save_results": False,
        "strategy_filtering": False,
        "error_detection_algorithms": "OD, RVD, RVD_orig",
    }
}

execution = 0 
configs = init(configs, execution)

# Or load the configuratoin parameters directly form the config.ini file
# configs = init(execution)

# Domain-Based Cell Folding
Domain-based cell folding in Matelda organizes cells from different tables by their semantic similarities.

In [4]:
# Set up multiprocessing pool (if necessary)
n_cores = configs["n_cores"]
pool = multiprocessing.Pool(n_cores)

# Call domain_based_folding
table_grouping_dict, table_size_dict = domain_based_folding(configs, pool)
# print(table_grouping_dict)

I need at least 2 labeled cells per table group to work at all and at least 2 * 6 labeled cells per table group to work effectively! Thant means you need to label 60 cells if you want reasonable (!) results:


# Quality-Based Cell Folding

In [5]:
# Use the keys returned by init (flattened names, not nested dictionaries)
cg_enabled = configs["column_grouping_enabled"]
column_groups_dir = os.path.join(configs["mediate_files_path"], "col_grouping_res", "col_df_res")
expected_file = os.path.join(column_groups_dir, "col_df_labels_cluster_0.pickle")

if cg_enabled and os.path.exists(expected_file):
    # Load the column grouping results to get the cluster sizes and the path to the column groups file.
    # This function returns a tuple of (number_of_col_clusters, cluster_sizes_dict, column_groups_df_path)
    _, cluster_sizes_dict, column_groups_df_path = loading_columns_grouping_results(
        table_grouping_dict, configs["mediate_files_path"]
    )

# Assuming column_groups_df_path and cluster_sizes_dict have been set by your domain-based folding cell:
if column_groups_df_path is not None:
    results = quality_based_folding(configs, pool, column_groups_df_path, cluster_sizes_dict)
    
    # Unpack the results
    (
        y_test_all, 
        y_local_cell_ids, 
        predicted_all, 
        y_labeled_by_user_all,
        unique_cells_local_index_collection, 
        samples, 
        n_user_labeled_cells
    ) = results
    
    # Present the results in the notebook
    print("Number of user labeled cells (quality based folding):", n_user_labeled_cells)
else:
    print("Skipping quality based folding due to missing column grouping results.")


Number of user labeled cells (quality based folding): 27700


### Presenting results from Quality-Based Cell Folding to user

In [7]:
###############
# Problem:    #
###############
# - we discussed last time that I should look into the pickle files which are generated in error_detection
# - not sure what to visualize and what each column is
###############

# Load the DataFrame from your pickle file
df = pd.read_pickle('/home/julian/projects/Matelda/output_quintet/output_quintet_0/_test_edbt_Quintet_27700_labels/cell_clustering/all_cell_clusters_records.pickle')
#df = pd.read_pickle('/home/julian/projects/Matelda/output_quintet/output_quintet_0/_test_edbt_Quintet_27700_labels/cell_clustering/cell_cluster_cells_dict_all.pickle')
#columns_to_show = ['table_cluster', 'col_cluster', 'n_cells', 'cells_per_cluster']
#df_sample = df[columns_to_show].head(5)
df_sample = df.head(5)
df_sample

,table_cluster,col_cluster,n_cells,n_init_labels,n_produced_cell_clusters,n_current_requiered_labels,remaining_labels,cells_per_cluster,errors_per_cluster,n_labels_updated
42,4,14,7390,1022,110,110,912,"{18: [0, 5, 6, 13, 15, 17, 18, 29, 31, 34, 41,...","{18: 5, 30: 2, 67: 15, 493: 0, 4: 12, 7: 2, 42...",491
33,4,2,7390,1022,104,104,918,"{112: [0, 12, 23, 136, 162, 181, 213, 238, 268...","{112: 0, 58: 0, 275: 0, 85: 0, 39: 0, 99: 0, 4...",485
52,2,2,2410,335,99,99,236,"{56: [0, 712, 906, 907, 913, 1032, 1033, 1034,...","{56: 0, 167: 0, 39: 0, 47: 0, 92: 61, 179: 0, ...",480
43,4,15,7390,1022,97,97,925,"{11: [0, 6, 24, 53, 63, 76, 116, 131, 175, 178...","{11: 0, 15: 0, 151: 0, 20: 0, 17: 0, 14: 0, 20...",478
23,1,2,2376,330,91,91,239,"{106: [0, 3, 6, 7, 9, 22, 26, 27, 28, 38, 42, ...","{106: 14, 148: 11, 10: 150, 96: 25, 78: 123, 1...",472


In [ ]:
####################
# Test             #
####################
# - tested the ipywidgets expand/collapse buttons
# - but thats probably not what we want to visualize
####################

# Load the DataFrame from your pickle file.
df = pd.read_pickle('/home/julian/projects/Matelda/output_quintet/output_quintet_0/_test_edbt_Quintet_27700_labels/cell_clustering/all_cell_clusters_records.pickle')

# Choose the columns you want to display and take a small sample.
columns_to_show = ['table_cluster', 'col_cluster', 'n_cells', 'cells_per_cluster']
df_sample = df[columns_to_show].head(5).copy()  # Using head(5) to keep it light.

def format_cells_per_cluster(cell_cluster):
    """
    Convert the cells_per_cluster dictionary into a multi-line string.
    For each key, show only the first 10 items of the list (if the list is long).
    """
    if isinstance(cell_cluster, dict):
        lines = []
        for key, value in cell_cluster.items():
            # Show a preview of the list; adjust the number (10 here) as needed.
            if isinstance(value, list) and len(value) > 10:
                preview = value[:10]
                lines.append(f"{key}: {preview} ... ({len(value)} items)")
            else:
                lines.append(f"{key}: {value}")
        return "\n".join(lines)
    return str(cell_cluster)

# Create an accordion widget for each row in the sample.
accordion_items = []
for idx, row in df_sample.iterrows():
    # Build a summary header for the accordion.
    header = f"Cluster: ({row['table_cluster']}, {row['col_cluster']}), n_cells: {row['n_cells']}"
    
    # Format the full details of cells_per_cluster.
    details = format_cells_per_cluster(row['cells_per_cluster'])
    
    # Create an output widget to hold the formatted details (using <pre> to preserve formatting).
    detail_output = widgets.HTML(value=f"<pre>{details}</pre>")
    
    # Create an Accordion widget with this detail.
    accordion = widgets.Accordion(children=[detail_output])
    accordion.set_title(0, header)
    
    accordion_items.append(accordion)

# Display all the accordions vertically.
display(widgets.VBox(accordion_items))
